In [1]:
!pip install -q desbordante pandas numpy

In [2]:
from google.colab import files
files.upload()

Saving reproduce_hyfd.py to reproduce_hyfd.py


{'reproduce_hyfd.py': b'"""\nWhat this script does\n  1. Runs HyFD (Desbordante\'s implementation) on each CSV and records #FDs and runtime.\n     Optionally also runs baselines (--baselines FUN FDep ...), which are algorithms from the paper.\n  2. Cross-checks the FD count of every small dataset with an independent brute-force\n     discoverer written in this file (--naive), to validate the library\'s output.\n  3. Prints a comparison table against the numbers reported in the paper\'s Table 1.\n\nNull semantics: the paper uses null = null. Nulls (empty cells, \'?\', \'NA\', \'NULL\') are replaced\nby one fixed token before running anything, so every tool sees them as an ordinary equal value.\n\nSetup:    pip install desbordante pandas numpy\nUsage:    python reproduce_hyfd.py --selftest\n          python reproduce_hyfd.py --data_dir ./data --datasets iris chess abalone --naive\n          python reproduce_hyfd.py --data_dir ./data --datasets iris abalone --baselines FUN FDep --runs 3 -

In [3]:
!python reproduce_hyfd.py --selftest

Class(Teacher, Subject) minimal FDs (brute force): set() (expected: empty set)
planted-FD table, brute force: ['{A,B}->C', '{A,C}->B', '{A}->D', '{B,C}->A', '{B,C}->D']
Desbordante HyFD: 5 FDs in 0.002s; naive: 5 FDs; identical sets: True


In [4]:
import os, urllib.request
os.makedirs("data", exist_ok=True)
BASE = "https://hpi.de/oldsite/fileadmin/user_upload/fachgebiete/naumann/projekte/repeatability/DataProfiling/FD_Datasets/"
files = {
    "iris": "iris.csv", "balance-scale": "balance-scale.csv", "chess": "chess.csv",
    "abalone": "abalone.csv", "nursery": "nursery.csv",
    "breast-cancer": "breast-cancer-wisconsin.csv", "bridges": "bridges.csv",
    "echocardiogram": "echocardiogram.csv", "adult": "adult.csv", "letter": "letter.csv",
    "ncvoter": "ncvoter_1001r_19c.csv", "hepatitis": "hepatitis.csv", "horse": "horse.csv",
}
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"}
failed = []
for name, f in files.items():
    try:
        req = urllib.request.Request(BASE + f, headers=headers)
        with urllib.request.urlopen(req, timeout=60) as r, open(f"data/{name}.csv", "wb") as out:
            out.write(r.read())
        print("downloaded", name)
    except Exception as e:
        print("FAILED", name, e)
        failed.append(name)
print("failed:", failed)

downloaded iris
downloaded balance-scale
downloaded chess
downloaded abalone
downloaded nursery
downloaded breast-cancer
downloaded bridges
downloaded echocardiogram
downloaded adult
downloaded letter
downloaded ncvoter
downloaded hepatitis
downloaded horse
failed: []


In [5]:
import pandas as pd
for n in files:
    df = pd.read_csv(f"data/{n}.csv", header=None, dtype=str, keep_default_na=False)
    print(n, df.shape, list(df.iloc[0])[:4])

iris (150, 5) ['5.1', '3.5', '1.4', '0.2']
balance-scale (625, 5) ['B', '1', '1', '1']
chess (28056, 7) ['a', '1', 'b', '3']
abalone (4177, 9) ['M', '0.455', '0.365', '0.095']
nursery (12960, 9) ['usual', 'proper', 'complete', '1']
breast-cancer (699, 11) ['1000025', '5', '1', '1']
bridges (108, 13) ['E1', 'M', '3', '1818']
echocardiogram (132, 13) ['11', '0', '71', '0']
adult (32561, 1) ['39;State-gov;77516;Bachelors;13;Never-married;Adm-clerical;Not-in-family;White;Male;2174;0;40;United-States;<=50K']
letter (20000, 17) ['T', '2', '8', '3']
ncvoter (1001, 19) ['voter_id', 'voter_reg_num', 'name_prefix', 'first_name']
hepatitis (155, 20) ['2', '30', '2', '1']
horse (300, 1) ['2;1;530101;38.50;66;28;3;3;?;2;5;4;4;?;?;?;3;5;45.00;8.40;?;?;2;2;11300;00000;00000;2;']


In [6]:
!python reproduce_hyfd.py --selftest

Class(Teacher, Subject) minimal FDs (brute force): set() (expected: empty set)
planted-FD table, brute force: ['{A,B}->C', '{A,C}->B', '{A}->D', '{B,C}->A', '{B,C}->D']
Desbordante HyFD: 5 FDs in 0.002s; naive: 5 FDs; identical sets: True


In [7]:
import pandas as pd
for n in files:
    df = pd.read_csv(f"data/{n}.csv", header=None, dtype=str, keep_default_na=False)
    print(n, df.shape, list(df.iloc[0])[:4])

iris (150, 5) ['5.1', '3.5', '1.4', '0.2']
balance-scale (625, 5) ['B', '1', '1', '1']
chess (28056, 7) ['a', '1', 'b', '3']
abalone (4177, 9) ['M', '0.455', '0.365', '0.095']
nursery (12960, 9) ['usual', 'proper', 'complete', '1']
breast-cancer (699, 11) ['1000025', '5', '1', '1']
bridges (108, 13) ['E1', 'M', '3', '1818']
echocardiogram (132, 13) ['11', '0', '71', '0']
adult (32561, 1) ['39;State-gov;77516;Bachelors;13;Never-married;Adm-clerical;Not-in-family;White;Male;2174;0;40;United-States;<=50K']
letter (20000, 17) ['T', '2', '8', '3']
ncvoter (1001, 19) ['voter_id', 'voter_reg_num', 'name_prefix', 'first_name']
hepatitis (155, 20) ['2', '30', '2', '1']
horse (300, 1) ['2;1;530101;38.50;66;28;3;3;?;2;5;4;4;?;?;?;3;5;45.00;8.40;?;?;2;2;11300;00000;00000;2;']


In [8]:
!python reproduce_hyfd.py --data_dir ./data --datasets iris balance-scale abalone breast-cancer bridges echocardiogram --no_header iris balance-scale abalone breast-cancer bridges echocardiogram --naive --baselines FUN FDep DFD --runs 3 --out results_small.csv

running iris (5 cols x 150 rows) ...
running balance-scale (5 cols x 625 rows) ...
running abalone (9 cols x 4177 rows) ...
running breast-cancer (11 cols x 699 rows) ...
running bridges (13 cols x 108 rows) ...
running echocardiogram (13 cols x 132 rows) ...

       dataset |           cols |           rows |       FDs ours |      FDs paper |    HyFD s ours |   HyFD s paper |     FUN s ours |    FUN s paper |    FDep s ours |   FDep s paper |     DFD s ours |    DFD s paper |      naive FDs |       same set
          iris |              5 |            150 |              4 |              4 |           0.00 |            0.1 |           0.00 |            0.1 |           0.00 |            0.1 |           0.00 |            0.2 |              4 |           True
 balance-scale |              5 |            625 |              1 |              1 |           0.00 |            0.1 |           0.00 |            0.1 |           0.01 |            0.2 |           0.00 |            0.3 |             

In [9]:
!python reproduce_hyfd.py --data_dir ./data --datasets hepatitis --no_header hepatitis --runs 1 --out results_hepatitis.csv

running hepatitis (20 cols x 155 rows) ...

       dataset |           cols |           rows |       FDs ours |      FDs paper |    HyFD s ours |   HyFD s paper |      naive FDs |       same set
     hepatitis |             20 |            155 |           8250 |           8250 |           0.26 |            0.6 |              - |              -

saved results_hepatitis.csv


In [10]:
!python reproduce_hyfd.py --data_dir ./data --datasets chess nursery --no_header chess nursery --naive --baselines FUN DFD --runs 1 --out results_chess_nursery.csv

running chess (7 cols x 28056 rows) ...
running nursery (9 cols x 12960 rows) ...

       dataset |           cols |           rows |       FDs ours |      FDs paper |    HyFD s ours |   HyFD s paper |     FUN s ours |    FUN s paper |     DFD s ours |    DFD s paper |      naive FDs |       same set
         chess |              7 |          28056 |              1 |              1 |           0.07 |            0.2 |           0.25 |            1.1 |           0.07 |            1.0 |              1 |           True
       nursery |              9 |          12960 |              1 |              1 |           0.04 |            0.5 |           0.47 |            1.8 |           0.06 |            0.9 |              1 |           True

saved results_chess_nursery.csv


In [11]:
!python reproduce_hyfd.py --data_dir ./data --datasets letter --no_header letter --baselines DFD --runs 1 --out results_letter.csv

running letter (17 cols x 20000 rows) ...

       dataset |           cols |           rows |       FDs ours |      FDs paper |    HyFD s ours |   HyFD s paper |     DFD s ours |    DFD s paper |      naive FDs |       same set
        letter |             17 |          20000 |             61 |             61 |           3.95 |            3.4 |           1.57 |            6.0 |              - |              -

saved results_letter.csv


In [12]:
!python reproduce_hyfd.py --data_dir ./data --datasets ncvoter --baselines FUN DFD --runs 1 --out results_ncvoter.csv

running ncvoter (19 cols x 1000 rows) ...

       dataset |           cols |           rows |       FDs ours |      FDs paper |    HyFD s ours |   HyFD s paper |     FUN s ours |    FUN s paper |     DFD s ours |    DFD s paper |      naive FDs |       same set
       ncvoter |             19 |           1000 |            758 |            758 |           0.06 |            0.4 |          56.45 |            4.0 |           1.09 |            5.1 |              - |              -

saved results_ncvoter.csv


In [17]:
!python reproduce_hyfd.py --data_dir ./data --datasets adult --no_header adult --baselines FUN DFD --runs 1 --out results_adult.csv

running adult (15 cols x 32561 rows) ...

       dataset |           cols |           rows |       FDs ours |      FDs paper |    HyFD s ours |   HyFD s paper |     FUN s ours |    FUN s paper |     DFD s ours |    DFD s paper |      naive FDs |       same set
         adult |             15 |          32561 |             78 |             78 |           1.28 |            1.1 |         279.18 |          111.6 |           0.88 |            5.9 |              - |              -

saved results_adult.csv


In [18]:
!python reproduce_hyfd.py --data_dir ./data --datasets chess nursery --no_header chess nursery --naive --baselines FUN DFD --runs 1 --out results_chess_nursery.csv

running chess (7 cols x 28056 rows) ...
running nursery (9 cols x 12960 rows) ...

       dataset |           cols |           rows |       FDs ours |      FDs paper |    HyFD s ours |   HyFD s paper |     FUN s ours |    FUN s paper |     DFD s ours |    DFD s paper |      naive FDs |       same set
         chess |              7 |          28056 |              1 |              1 |           0.07 |            0.2 |           0.25 |            1.1 |           0.10 |            1.0 |              1 |           True
       nursery |              9 |          12960 |              1 |              1 |           0.04 |            0.5 |           0.46 |            1.8 |           0.07 |            0.9 |              1 |           True

saved results_chess_nursery.csv


In [19]:
!lscpu | grep "Model name"
!free -h
!python --version
!pip show desbordante | head -3

Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.2Gi       8.4Gi       2.3Mi       3.5Gi        11Gi
Swap:             0B          0B          0B
Python 3.13.16
Name: desbordante
Version: 2.5.0
Summary: Science-intensive high-performance data profiler
ERROR: Pipe to stdout was broken


In [21]:
from google.colab import files
for f in ["results_small.csv", "results_hepatitis.csv", "results_letter.csv",
          "results_ncvoter.csv", "results_adult.csv", "results_chess_nursery.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>